# Proyecto Final Notebook Analítico con SQL y Python
**Estudiante:** Sergio Renato Cedeño Zavaleta
**Módulo:** Introducción a SQL y Python para Análisis de Datos- Data Analytics DMC Perú

**Título:** Análisis Exploratorio de Gastos Reales del Proyecto Inmobiliario PIANA

**Fuente del dataset:** Dataset propio correspondiente a los gastos reales registrados durante la ejecución del proyecto inmobiliario PIANA.

**Objetivo:** Desarrollar un análisis exploratorio de los gastos reales del proyecto mediante Python, pandas, visualizaciones interactivas con Plotly y consultas SQL utilizando DuckDB, con el objetivo de identificar patrones de gasto, distribución de costos y principales componentes económicos del proyecto.

In [195]:
!pip install duckdb openpyxl -q

import pandas as pd
import numpy as np
import duckdb
import plotly.express as px

# Configurar formato de números con separador de miles
pd.options.display.float_format = '{:,.2f}'.format

In [196]:
df = pd.read_excel("/content/Gastos de Proyecto Piana.xlsx")

## Exploración inicial

En esta sección se realiza una exploración preliminar del dataset con el objetivo de comprender su estructura, dimensiones, nombres de columnas y tipos de datos presentes en el conjunto de información.

In [197]:
df.shape

(3533, 13)

In [198]:
df.head()

,FECHA,Item,DESCRIPCION,UM,CANTIDAD,P.UNIT,P.TOTAL,#FAC/BOL,PROVEEDOR,FASE,COMPONENTE,ESPECIALIDAD,CLASIFICACIÓN
0,2023-02-28,Item 1,TERRENO,glb,1.00,"1,814,400.00","1,814,400.00",S/C,TERRENO,PRE CONSTRUCCIÓN,TERRENO,TERRENO,TERRENO
1,2023-02-28,Item 2,ALCABALA,glb,1.00,"39,199.51","39,199.51",S/C,MPT,PRE CONSTRUCCIÓN,TERRENO,TERRENO,TERRENO
2,2024-01-10,Item 3,HORMIGÓN,glb,1.00,"3,947.40","3,947.40",S/C,LEON,CONSTRUCCIÓN,MATERIALES,ESTRUCTURA,COSTO DIRECTO
3,2024-01-11,Item 4,GASOLINA,GLB,1.00,150.00,150.00,S/C,N/D,PRE CONSTRUCCIÓN,GASTOS GEN.,OTROS,COSTO INDIRECTO
4,2024-01-11,Item 5,SUELDO ARQUITECTO,GLB,1.00,"1,200.00","1,200.00",S/C,SMITH GIRON,PRE CONSTRUCCIÓN,GASTOS GEN.,PERSONAL ADMINISTRATIVO,COSTO INDIRECTO


In [199]:
df.tail()

,FECHA,Item,DESCRIPCION,UM,CANTIDAD,P.UNIT,P.TOTAL,#FAC/BOL,PROVEEDOR,FASE,COMPONENTE,ESPECIALIDAD,CLASIFICACIÓN
3528,2026-07-04,Item 3535,COLOCACIÓN DE PUERTAS,GLB,1.00,"1,572.00","1,572.00",S/C,NICOLE GARCÍA,CONSTRUCCIÓN,MANO DE OBRA,ARQUITECTURA,COSTO DIRECTO
3529,2026-07-06,Item 3536,RICADONNA RUBY,UND,1.00,91.05,91.05,F516-276986,METRO,CONSTRUCCIÓN,GASTOS GEN.,OTROS,COSTO INDIRECTO
3530,2026-07-06,Item 3537,RICADONNA PPROSECCO X 750 ML,UND,12.00,71.91,862.92,F028-10832,LA TABERNA,CONSTRUCCIÓN,GASTOS GEN.,OTROS,COSTO INDIRECTO
3531,2026-07-08,Item 3538,LATEX PATO BLANCO X GL,GAL,1.00,55.50,55.50,F001-37768,PIALESA,CONSTRUCCIÓN,MATERIALES,ARQUITECTURA,COSTO DIRECTO
3532,2026-07-08,Item 3539,LATEX PATO GRIS CLARO X GL,GAL,1.00,55.50,55.50,F001-37768,PIALESA,CONSTRUCCIÓN,MATERIALES,ARQUITECTURA,COSTO DIRECTO


In [200]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3533 entries, 0 to 3532
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   FECHA          3533 non-null   datetime64[ns]
 1   Item           3533 non-null   object        
 2   DESCRIPCION    3533 non-null   object        
 3   UM             3533 non-null   object        
 4   CANTIDAD       3533 non-null   float64       
 5   P.UNIT         3533 non-null   float64       
 6   P.TOTAL        3533 non-null   float64       
 7   #FAC/BOL       3529 non-null   object        
 8   PROVEEDOR      3533 non-null   object        
 9   FASE           3533 non-null   object        
 10  COMPONENTE     3533 non-null   object        
 11  ESPECIALIDAD   3533 non-null   object        
 12  CLASIFICACIÓN  3533 non-null   object        
dtypes: datetime64[ns](1), float64(3), object(9)
memory usage: 358.9+ KB


In [201]:
df.columns

Index(['FECHA', 'Item', 'DESCRIPCION', 'UM', 'CANTIDAD', 'P.UNIT', 'P.TOTAL',
       '#FAC/BOL', 'PROVEEDOR', 'FASE', 'COMPONENTE', 'ESPECIALIDAD',
       'CLASIFICACIÓN'],
      dtype='object')

## Análisis de calidad de datos

En esta etapa se realizará la evaluación de la calidad del conjunto de datos mediante la identificación de valores nulos, valores vacíos, registros duplicados y posibles inconsistencias en la información, con el objetivo de asegurar que los datos sean adecuados para el análisis posterior.

In [202]:
df.isnull().sum()

,0
FECHA,0
Item,0
DESCRIPCION,0
UM,0
CANTIDAD,0
P.UNIT,0
P.TOTAL,0
#FAC/BOL,4
PROVEEDOR,0
FASE,0


In [203]:
(df == "").sum()

,0
FECHA,0
Item,0
DESCRIPCION,0
UM,0
CANTIDAD,0
P.UNIT,0
P.TOTAL,0
#FAC/BOL,0
PROVEEDOR,0
FASE,0


In [204]:
df.duplicated().sum()

np.int64(0)

In [205]:
df["PROVEEDOR"].unique()

array(['TERRENO', 'MPT', 'LEON', 'N/D', 'SMITH GIRON', 'PLANOS',
       'PAJARES', 'CORCUERA', 'SATT', 'HIDRANDINA', 'HABITAT',
       'SUPERVISOR', 'PROMART', '3A', 'DIST.ROYER', 'FACEBOOK', 'ADELMO',
       'GORDILLO', 'VIA SOLUTEC', 'TRENZADO PERUANO', 'OLINDA',
       'MONICA LIZETH', 'KIKE', 'SUNARP', 'RONNY SAC', 'VALDERRAMA',
       'BRYANCO', 'EDEL', 'JUAN URTEAGA GARCIA', 'MAQUINORT',
       'COLEGIO DE INGENIEROS', 'MANUEL FORTUANO', 'MANO DE OBRA PROPIA',
       'GLADITH HIDALGO', 'SEDALIB', 'EL DIAMANTE', 'HYDRELL', 'AKYVAJA',
       'ESTRELLA DE DAVID', 'DOUGLAS MALPIC', 'CARLOS GUZMÁN',
       'JOSUE CARBAJAL', 'NOE RUBIO', 'VASKEVEL', 'INKAFERRO', 'ULTRACOM',
       'GARCIA OBER', 'PROYECTISTA', 'S/P', 'ENNIS RAMIREZ',
       'ELIDA ROBLES', 'JANET RUIZ', 'E S LARCO', 'BURGOS & GUTIERREZ',
       'SINDICATO', 'JUAN GUTIERREZ', 'DELTRON', 'SUPERTEC', 'OBLITAS',
       'PINTEL', 'GILMER SALDAÑA', 'CHAN CHAN', 'CPPQ', 'MODESTO',
       'SOMOS PAPER', 'TAXI', 'RONALD OLIVARE

### Revisión de posibles inconsistencias

Se realizó una revisión de los valores únicos en las variables categóricas para identificar diferencias de escritura que puedan afectar la agrupación de datos.

Se encontraron inconsistencias en la columna PROVEEDOR debido a diferencias en el uso de mayúsculas/minúsculas y espacios adicionales. Algunos ejemplos identificados son:

- "Arco Techo" y "ARCO TECHO"
- "Movistar" y "MOVISTAR"
- "Integra" e "INTEGRA"
- "Prima" y "PRIMA"

Estas diferencias serán consideradas para una posterior estandarización de datos antes del análisis.

In [206]:
df["ESPECIALIDAD"].unique()

array(['TERRENO', 'ESTRUCTURA', 'OTROS', 'PERSONAL ADMINISTRATIVO',
       'GASTOS LEGALES/TÉCNICOS', 'CORREDORES', 'ELIMINACIÓN',
       'EQUIPOS Y HERRAMIENTAS', 'PUBLICIDAD', 'SERVICIO', 'ARQUITECTURA',
       'PLANILLA', 'GASTOS DE OFICINA', 'RESANES VECINO', 'ELÉCTRICA',
       'SANITARIA', "EPP'S", 'UTILIDAD'], dtype=object)

## Estadística descriptiva

En esta etapa se realizará un análisis descriptivo de las variables del conjunto de datos, evaluando las variables numéricas mediante medidas estadísticas y las variables categóricas mediante la descripción de sus valores y frecuencias.

In [207]:
df.describe()

,FECHA,CANTIDAD,P.UNIT,P.TOTAL
count,3533,"3,533.00","3,533.00","3,533.00"
mean,2025-10-24 14:27:44.987262976,49.52,"1,523.82","2,548.40"
min,2023-02-28 00:00:00,0.00,0.00,0.00
25%,2025-07-01 00:00:00,1.00,10.38,54.92
50%,2025-11-11 00:00:00,1.00,40.48,157.50
75%,2026-03-07 00:00:00,12.00,236.48,"1,128.60"
max,2026-07-08 00:00:00,"15,500.00","1,814,400.00","1,814,400.00"
std,NaN,359.96,"30,768.43","31,159.32"


### Interpretación de variables numéricas

La estadística descriptiva muestra que las variables CANTIDAD, P.UNIT y P.TOTAL presentan una alta dispersión debido a la existencia de valores máximos muy superiores al promedio. Esto indica la presencia de registros con montos o cantidades elevadas, los cuales deberán ser considerados durante el análisis posterior.

La diferencia entre la media y la mediana evidencia una distribución con posibles valores atípicos, especialmente en precios unitarios y totales.

In [208]:
df.describe(include="object")

,Item,DESCRIPCION,UM,#FAC/BOL,PROVEEDOR,FASE,COMPONENTE,ESPECIALIDAD,CLASIFICACIÓN
count,3533,3533,3533,3529,3533,3533,3533,3533,3533
unique,3533,1989,61,1206,398,4,7,18,5
top,Item 3539,GASOLINA,UND,S/C,DIST.ROYER,CONSTRUCCIÓN,MATERIALES,OTROS,COSTO DIRECTO
freq,1,151,1525,855,592,3439,1492,602,2079


## Visualizaciones con Plotly

Con el objetivo de analizar el comportamiento de las compras realizadas, se generaron visualizaciones interactivas utilizando la librería Plotly. Los gráficos permiten identificar la distribución de los montos, los principales proveedores según el gasto acumulado, la evolución de las compras en el tiempo y la presencia de posibles valores atípicos.

Para una mejor interpretación del comportamiento operativo, se excluyó el proveedor "TERRENO" en los análisis comparativos, debido a que corresponde a una adquisición extraordinaria que podría distorsionar la tendencia general de los gastos.

In [209]:
df["FECHA"] = pd.to_datetime(df["FECHA"])
df["AÑO"] = df["FECHA"].dt.year

In [210]:
gasto_fecha = (
    df[df["PROVEEDOR"] != "TERRENO"]
    .groupby("FECHA")["P.TOTAL"]
    .sum()
    .reset_index()
)

fig = px.line(
    gasto_fecha,
    x="FECHA",
    y="P.TOTAL",
    title="Evolución del gasto en el tiempo"
)

fig.show()

#### Evolución del gasto en el tiempo

El gráfico permite analizar el comportamiento del gasto durante el periodo evaluado, identificando los meses con mayores y menores niveles de compra. Las variaciones observadas reflejan cambios en las necesidades de abastecimiento y permiten reconocer periodos de mayor actividad operativa.


In [211]:
top_proveedores_filtrado = (
    df[~df["PROVEEDOR"].isin(["TERRENO", "UTILIDAD"])]
    .groupby("PROVEEDOR")["P.TOTAL"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig = px.bar(
    top_proveedores_filtrado,
    x="PROVEEDOR",
    y="P.TOTAL",
    title="Top 10 Proveedores por Monto Total de Compra"
)

fig.show()

#### Interpretación: Top 10 proveedores por monto total de compra

El gráfico permite identificar los 10 proveedores con mayor participación económica dentro del total de compras realizadas.

Los resultados muestran los proveedores que concentran una mayor proporción del gasto, permitiendo reconocer aquellos con mayor impacto financiero y que requieren mayor seguimiento dentro del proceso de abastecimiento.

Asimismo, se evidencia que el proveedor con mayor cantidad de registros no necesariamente representa el mayor monto acumulado de compra.

## Gasto por Especialidad

Aquí visualizaremos la distribución del gasto total por cada especialidad.

In [212]:
gasto_especialidad = (
    df[~df["ESPECIALIDAD"].isin(["TERRENO", "UTILIDAD"])]
    .groupby("ESPECIALIDAD")["P.TOTAL"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

fig = px.bar(
    gasto_especialidad,
    x="P.TOTAL",
    y="ESPECIALIDAD",
    orientation="h",
    title="Gasto Total por Especialidad"
)

fig.show()

### Interpretación: Gasto por especialidad

El gráfico permite analizar cómo se distribuye el gasto total de compras entre las diferentes especialidades registradas.

Las especialidades con mayor nivel de gasto representan los principales componentes de inversión y requieren un mayor seguimiento para asegurar un adecuado control de costos y una correcta planificación de recursos.


In [213]:
gasto_clasificacion_sin_terreno_utilidad = (
    df[~df['CLASIFICACIÓN'].isin(['TERRENO', 'UTILIDAD'])]
    .groupby("CLASIFICACIÓN")["P.TOTAL"]
    .sum()
    .reset_index()
)

fig = px.pie(
    gasto_clasificacion_sin_terreno_utilidad,
    values="P.TOTAL",
    names="CLASIFICACIÓN",
    title="Participación del gasto por Clasificación"
)

fig.show()

#### Participación del gasto por Clasificación

Este gráfico de pastel muestra la proporción del gasto total distribuido entre las categorías de clasificación de costos, excluyendo 'TERRENO' y 'UTILIDAD' para una visión más enfocada en los gastos operativos y de inversión.

In [214]:
df_boxplot = df[
    ~df["PROVEEDOR"].isin(["TERRENO", "UTILIDAD"])
]

fig = px.box(
    df_boxplot,
    y="P.TOTAL",
    title="Distribución de montos de compra"
)

fig.show()


#### Interpretación: Boxplot del monto total de compra

El gráfico permite analizar la distribución de los montos totales de compra e identificar posibles valores atípicos dentro de las operaciones registradas.

Para evitar distorsiones en la visualización, se excluyeron los conceptos "TERRENO" y "UTILIDAD", debido a que corresponden a valores extraordinarios que presentan montos significativamente diferentes al comportamiento habitual de las compras.

La distribución obtenida permite observar el rango donde se concentra la mayoría de las compras y detectar aquellas operaciones con montos superiores que requieren una revisión adicional.

In [215]:
# Filtrar el dataset para excluir UTILIDAD en FASE y TERRENO en COMPONENTE
df_filtrado = df[
    (df["FASE"] != "UTILIDAD") &
    (df["COMPONENTE"] != "TERRENO")
]

# Crear scatter plot animado
fig = px.scatter(
    df_filtrado,
    x="CANTIDAD",
    y="P.TOTAL",
    color="FASE",
    animation_frame="AÑO",
    animation_group="DESCRIPCION",
    title="Cantidad vs gasto total por fase",
    hover_data=["DESCRIPCION", "PROVEEDOR"],
    log_y=True
)

# Reducir velocidad de animación
fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 2000
fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 1000

fig.show()






El gráfico muestra la **relación entre la cantidad de unidades compradas (CANTIDAD) y el gasto total (P.TOTAL)**, diferenciando por fase del proyecto y excluyendo las categorías *UTILIDAD* y *TERRENO*. El eje Y está en **escala logarítmica**, lo que permite visualizar tanto gastos pequeños como grandes sin que los valores extremos distorsionen la distribución.

- La fase de **CONSTRUCCIÓN** es la que concentra la mayor actividad y gasto acumulado en operaciones medianas.  
- La fase de **PRE CONSTRUCCIÓN** refleja compras puntuales de alto valor, aunque menos frecuentes.  
- La fase de **COMERCIALIZACIÓN** requiere menor presupuesto en comparación con las otras.  



In [216]:
df["AÑO"] = pd.to_datetime(df["FECHA"]).dt.year

## Consultas SQL con DuckDB

Se utilizó DuckDB para realizar consultas SQL directamente sobre el DataFrame cargado, permitiendo analizar la información mediante filtros, agrupaciones, ordenamientos y cálculos sobre los datos de compras.

Las consultas realizadas buscan obtener información relevante sobre el comportamiento del gasto, proveedores y distribución de compras.

In [217]:
con = duckdb.connect()

In [218]:
con.register("datos", df_filtrado)

In [219]:
query_inspeccion = """
SELECT *
FROM datos
LIMIT 10
"""

resultado_inspeccion = con.execute(query_inspeccion).df()

resultado_inspeccion

,FECHA,Item,DESCRIPCION,UM,CANTIDAD,P.UNIT,P.TOTAL,#FAC/BOL,PROVEEDOR,FASE,COMPONENTE,ESPECIALIDAD,CLASIFICACIÓN,AÑO
0,2024-01-10,Item 3,HORMIGÓN,glb,1.00,"3,947.40","3,947.40",S/C,LEON,CONSTRUCCIÓN,MATERIALES,ESTRUCTURA,COSTO DIRECTO,2024
1,2024-01-11,Item 4,GASOLINA,GLB,1.00,150.00,150.00,S/C,N/D,PRE CONSTRUCCIÓN,GASTOS GEN.,OTROS,COSTO INDIRECTO,2024
2,2024-01-11,Item 5,SUELDO ARQUITECTO,GLB,1.00,"1,200.00","1,200.00",S/C,SMITH GIRON,PRE CONSTRUCCIÓN,GASTOS GEN.,PERSONAL ADMINISTRATIVO,COSTO INDIRECTO,2024
3,2024-01-17,Item 6,PLANOS,GLB,1.00,"6,975.00","6,975.00",S/C,PLANOS,PRE CONSTRUCCIÓN,GASTOS GEN.,GASTOS LEGALES/TÉCNICOS,COSTO INDIRECTO,2024
4,2024-02-04,Item 7,HORMIGÓN,m3,153.00,36.03,"5,512.05",S/C,LEON,CONSTRUCCIÓN,MATERIALES,ESTRUCTURA,COSTO DIRECTO,2024
5,2024-02-27,Item 8,PROYECTO,glb,1.00,"24,000.00","24,000.00",S/C,PAJARES,PRE CONSTRUCCIÓN,GASTOS GEN.,GASTOS LEGALES/TÉCNICOS,COSTO INDIRECTO,2024
6,2024-02-28,Item 9,GASTOS NOTARIALES,glb,1.00,"8,437.50","8,437.50",S/C,CORCUERA,PRE CONSTRUCCIÓN,GASTOS GEN.,GASTOS LEGALES/TÉCNICOS,COSTO INDIRECTO,2024
7,2024-06-19,Item 10,TRIBUTOS MUNICIPALES,glb,1.00,486.86,486.86,S/C,SATT,PRE CONSTRUCCIÓN,GASTOS GEN.,GASTOS LEGALES/TÉCNICOS,COSTO INDIRECTO,2024
8,2024-06-21,Item 11,SANEAMIENTO DE SERVICIO ELÉCTRICO,glb,1.00,119.85,119.85,S/C,HIDRANDINA,PRE CONSTRUCCIÓN,GASTOS GEN.,GASTOS LEGALES/TÉCNICOS,COSTO INDIRECTO,2024
9,2024-07-03,Item 12,COMISION VENTA,UND,1.00,"8,732.82","8,732.82",S/C,HABITAT,COMERCIALIZACIÓN,GASTOS GEN.,CORREDORES,GASTOS DE VENTA,2024


In [220]:
query_filtro = """
SELECT
    DESCRIPCION,
    PROVEEDOR,
    "P.TOTAL"
FROM datos
WHERE FASE = 'CONSTRUCCIÓN'
ORDER BY "P.TOTAL" DESC
LIMIT 10
"""

resultado_filtro = con.execute(query_filtro).df()

resultado_filtro

,DESCRIPCION,PROVEEDOR,P.TOTAL
0,"FIERRO 3/8""",PROMART,"122,659.68"
1,CONCRETO PREMEZCLADO 280 MS H67 A5 CON BOMBA,OBLITAS,"100,146.60"
2,ASCENSOR,EDEL,"75,082.50"
3,TABLON CEDRO MIEL 20 X 100,DECOR IMPORT,"72,000.00"
4,ASCENSOR,EDEL,"62,568.75"
5,PUERTA DORMITORIO VANO 800 X 2100,ARES,"57,207.38"
6,"FIERRO 3/8""",INKAFERRO,"55,713.84"
7,"FIERRO 5/8""",INKAFERRO,"55,650.38"
8,ASCENSOR,EDEL,"54,112.50"
9,"FIERRO 3/8""",INKAFERRO,"53,605.57"


In [221]:
query1 = """
SELECT
    FASE,
    SUM("P.TOTAL") AS GASTO_TOTAL
FROM datos
GROUP BY FASE
ORDER BY GASTO_TOTAL DESC
"""

resultado1 = con.execute(query1).df()


resultado1

,FASE,GASTO_TOTAL
0,CONSTRUCCIÓN,"6,767,645.68"
1,PRE CONSTRUCCIÓN,"167,666.31"
2,COMERCIALIZACIÓN,"162,073.77"


In [222]:
query2 = """
SELECT
    PROVEEDOR,
    SUM("P.TOTAL") AS GASTO_TOTAL
FROM datos
GROUP BY PROVEEDOR
ORDER BY GASTO_TOTAL DESC
LIMIT 10
"""

resultado2 = con.execute(query2).df()

resultado2

,PROVEEDOR,GASTO_TOTAL
0,MANO DE OBRA PROPIA,"867,937.50"
1,OBLITAS,"810,173.66"
2,INKAFERRO,"406,216.45"
3,PROMART,"381,888.84"
4,ZURDO,"359,293.48"
5,CECILIA DELFIN,"274,732.52"
6,ARCO TECHO,"265,620.47"
7,EDEL,"252,190.50"
8,MODESTO,"177,814.75"
9,FANNY VASQUEZ,"157,500.23"


In [223]:
query3 = """
SELECT
    AÑO,
    SUM("P.TOTAL") AS GASTO_ANUAL
FROM datos
GROUP BY AÑO
ORDER BY AÑO
"""

resultado3 = con.execute(query3).df()

resultado3

,AÑO,GASTO_ANUAL
0,2024,"648,206.61"
1,2025,"4,888,204.35"
2,2026,"1,560,974.79"


In [224]:
query4 = """
SELECT
    COMPONENTE,
    SUM("P.TOTAL") AS GASTO_TOTAL
FROM datos
GROUP BY COMPONENTE
ORDER BY GASTO_TOTAL DESC
LIMIT 10
"""

resultado4 = con.execute(query4).df()

resultado4

,COMPONENTE,GASTO_TOTAL
0,MATERIALES,"3,129,502.65"
1,SUB CONTRATO,"1,736,554.62"
2,GASTOS GEN.,"1,151,988.96"
3,MANO DE OBRA,"1,040,723.40"
4,EQUIPOS Y HERRAMIENTAS,"38,616.13"


In [225]:
query5 = """
SELECT
    FASE,
    SUM(CANTIDAD) AS CANTIDAD_TOTAL,
    AVG("P.TOTAL") AS GASTO_PROMEDIO
FROM datos
GROUP BY FASE
ORDER BY GASTO_PROMEDIO DESC
"""

resultado5 = con.execute(query5).df()

resultado5


,FASE,CANTIDAD_TOTAL,GASTO_PROMEDIO
0,COMERCIALIZACIÓN,34.00,"8,103.69"
1,PRE CONSTRUCCIÓN,390.00,"2,502.48"
2,CONSTRUCCIÓN,"174,533.89","1,967.91"
